# Training Ingestion Hardening & Event Lifecycle Guide

This notebook teaches the complete implementation of a production-grade training pipeline with:
- Robust data ingestion (local + Roboflow merge)
- Training event lifecycle tracking
- Kafka-ready, decoupled event publishing
- SOLID principles applied throughout

**Audience**: Developers, ML engineers, system architects

**Prerequisites**: Python 3.8+, SQLite3, basic understanding of ML pipelines

---

## Part 1: Understanding the Problem

### The Challenge

Before this implementation, the training system had several issues:

1. **Duplicate Event Logging**
   - Streamlit pages created training events
   - Then `scripts/train.py` created another event
   - Result: one user action → two DB rows

2. **No Status Lifecycle**
   - Events inserted as `pending` and never updated
   - Dashboard metrics (success rate, running count) were unreliable

3. **Weak Ingestion Pipeline**
   - `--source roboflow` failed silently if credentials missing
   - No true merge of Roboflow + local samples
   - No deduplication or validation

4. **Monolithic Event Handling**
   - No abstraction for event publishing
   - Hard to extend to Kafka or other backends

### The Solution Architecture

```
Streamlit UI → AnalyticsDB (create event_id)
           → scripts/train.py (reuse event_id, update status)
           → PipelineEventPublisher (abstraction: noop or Kafka)
```

---

## Part 2: Training Event Lifecycle

### Concept: Event State Machine

A training event follows a strict lifecycle:

```
pending → running → completed
       ↘           ↙
         failed
```

### Implementation in AnalyticsDB

In [3]:
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

In [8]:
os.getcwd()

'd:\\Thomas\\PERSONAL\\Projects\\vision-ml-system'

In [7]:
os.chdir("d:\\Thomas\\PERSONAL\\Projects\\vision-ml-system")

In [9]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from vision_ml.analytics.analytics_db import AnalyticsDB

db = AnalyticsDB('data/analytics_demo.db')

event_id = db.save_training_event({
    'trigger_type': 'manual',
    'dataset_size': 500,
    'drift_score': 0.0,
    'model_version': 'v2.0',
    'status': 'pending',
})

print(f'Created event: {event_id}')
events = db.get_training_events(limit=1)
print(f'Status: {events[0]["status"]}')

Created event: train_20260318_083426
Status: pending


In [10]:
db.mark_training_event_running(event_id, dataset_size=520)
events = db.get_training_events(limit=1)
print(f'Status: {events[0]["status"]}')
print(f'Dataset size: {events[0]["dataset_size"]}')

Status: running
Dataset size: 520


In [11]:
db.mark_training_event_completed(event_id)
events = db.get_training_events(limit=1)
print(f'Status: {events[0]["status"]}')

Status: completed


### Key Design Pattern: Reusable Event IDs

**Problem**: Duplicate events from page + script.

**Solution**: Pass `event_id` via CLI.

```python
# Streamlit page
event_id = db.save_training_event({...})
subprocess.Popen([sys.executable, 'scripts/train.py', '--event-id', event_id])

# Script
if args.event_id:
    db.update_training_event_status(args.event_id, 'pending')
```

---

## Part 3: Data Ingestion Pipeline

### The Three Modes

| Mode | Source | Behavior |
|------|--------|----------|
| `local` | Local pseudo-labels | Collect and build |
| `roboflow` | Roboflow API | Download or fail |
| `both` | Local + Roboflow | Merge with dedup |

### Deduplication by Filename

In [13]:
from scripts.prepare_data import _dedupe_samples

samples = [
    ('/local/frame_001.jpg', {'boxes': [[1, 1, 10, 10]], 'class_ids': [0]}),
    ('/roboflow/frame_001.jpg', {'boxes': [[2, 2, 11, 11]], 'class_ids': [0]}),
    ('/local/frame_002.jpg', {'boxes': [], 'class_ids': []}),
]

print(f'Before: {len(samples)} samples')
deduped = _dedupe_samples(samples)
print(f'After: {len(deduped)} samples')
print('Duplicates removed, first-seen order preserved')

Before: 3 samples
After: 2 samples
Duplicates removed, first-seen order preserved


### Sample Validation

In [14]:
from scripts.prepare_data import _validate_samples
import tempfile
import cv2
import numpy as np

with tempfile.TemporaryDirectory() as tmpdir:
    valid_img = f'{tmpdir}/valid.jpg'
    cv2.imwrite(valid_img, np.zeros((100, 100, 3), dtype=np.uint8))
    
    samples = [
        (valid_img, {'boxes': [[10, 10, 50, 50]], 'class_ids': [0]}),
        (f'{tmpdir}/missing.jpg', {'boxes': [], 'class_ids': []}),
        (valid_img, {'boxes': [[1, 1, 20, 20]], 'class_ids': []}),
    ]
    
    print(f'Before: {len(samples)} samples')
    validated = _validate_samples(samples)
    print(f'After: {len(validated)} samples')
    print('Invalid dropped, missing classes auto-filled')

Before: 3 samples
2026-03-18 08:50:25 | WARNING  | scripts.prepare_data | Dropped 1 invalid samples during validation
After: 2 samples
Invalid dropped, missing classes auto-filled


---

## Part 4: Event Publishing (Kafka-Ready)

### The Publisher Abstraction

**Principle**: Depend on abstraction, not concrete implementation (Dependency Inversion).

```python
class PipelineEventPublisher(ABC):
    @abstractmethod
    def publish(self, event_name: str, payload: Dict) -> None:
        pass
```

**Implementations**:
- `NoopPipelineEventPublisher`: Default (no-op)
- `KafkaPipelineEventPublisher`: Optional Kafka backend

In [18]:
from vision_ml.events.publishers import get_pipeline_event_publisher

config = {'events': {'backend': 'noop'}}
publisher = get_pipeline_event_publisher(config)

publisher.publish('training.started', {
    'event_id': 'train_20260317_211500',
    'trigger': 'manual',
    'run_name': 'demo_run',
})

print('Event published (noop backend)')
publisher

Event published (noop backend)


### Lifecycle Events in scripts/train.py

```python
publisher.publish('training.started', {...})

try:
    trainer.train()
except Exception:
    publisher.publish('training.failed', {...})
    raise
else:
    publisher.publish('training.completed', {...})
```

---

## Part 5: SOLID Principles Applied

### 1. Single Responsibility Principle (SRP)

Each class/function has one reason to change:

- `AnalyticsDB`: Manages training event persistence
- `PipelineEventPublisher`: Publishes events
- `_dedupe_samples()`: Deduplicates samples
- `_validate_samples()`: Validates samples

**Bad** (violates SRP):
```python
def prepare_data_and_publish_and_log():
    # Does ingestion, publishing, AND logging
    pass
```

**Good** (respects SRP):
```python
def prepare_data():
    samples = collect_samples()
    samples = _dedupe_samples(samples)
    samples = _validate_samples(samples)
    publisher.publish('ingestion.completed', {...})
```

### 2. Open/Closed Principle (OCP)

Open for extension, closed for modification.

**Example**: Event publishers

```python
# Closed for modification: PipelineEventPublisher interface is stable
class PipelineEventPublisher(ABC):
    @abstractmethod
    def publish(self, event_name: str, payload: Dict) -> None:
        pass

# Open for extension: Add new publishers without changing existing code
class KafkaPipelineEventPublisher(PipelineEventPublisher):
    def publish(self, event_name: str, payload: Dict) -> None:
        self._producer.send(self.topic, {...})

class RabbitMQPublisher(PipelineEventPublisher):
    def publish(self, event_name: str, payload: Dict) -> None:
        self._channel.basic_publish(...)
```

### 3. Liskov Substitution Principle (LSP)

Subtypes must be safely substitutable for their base types.

```python
# Both publishers have the same contract
publisher: PipelineEventPublisher = get_pipeline_event_publisher(config)

# Works the same whether it's Noop or Kafka
publisher.publish('training.started', {...})
```

### 4. Interface Segregation Principle (ISP)

Prefer small, focused interfaces over fat ones.

**Bad** (fat interface):
```python
class TrainingManager:
    def ingest_data(self): ...
    def train(self): ...
    def publish_event(self): ...
    def log_metrics(self): ...
    def register_model(self): ...
```

**Good** (segregated interfaces):
```python
class DataIngester:
    def ingest(self): ...

class Trainer:
    def train(self): ...

class EventPublisher:
    def publish(self, event, payload): ...
```

### 5. Dependency Inversion Principle (DIP)

Depend on abstractions, not concrete implementations.

```python
# Bad: Depends on concrete Kafka class
def train(config):
    publisher = KafkaPublisher(...)
    publisher.publish(...)

# Good: Depends on abstraction
def train(config):
    publisher = get_pipeline_event_publisher(config)  # Returns abstraction
    publisher.publish(...)  # Works with any implementation
```

---

## Part 6: Other Key Design Principles

### DRY (Don't Repeat Yourself)

**Applied**: Deduplication logic in `_dedupe_samples()` is reused for both local and Roboflow sources.

### KISS (Keep It Simple, Stupid)

**Applied**: Event lifecycle uses simple state machine (pending → running → completed/failed).

### Separation of Concerns

**Applied**: Ingestion, training, and event publishing are separate modules.

### High Cohesion, Low Coupling

**Applied**: 
- `AnalyticsDB` handles only DB operations
- `PipelineEventPublisher` handles only event publishing
- They communicate via simple interfaces

### Idempotency

**Applied**: Passing `--event-id` allows safe retries without duplicate events.

### Fail Fast + Explicit Errors

**Applied**: `--source roboflow` with missing credentials exits with code 1 (not silent failure).

---

## Part 7: Complete Workflow Example

### Step 1: User clicks "Start Training" in Streamlit

In [19]:
# Simulating pages/4_training.py behavior

db = AnalyticsDB('data/analytics_demo.db')

event_id = db.save_training_event({
    'trigger_type': 'manual',
    'dataset_size': 500,
    'drift_score': 0.0,
    'model_version': 'v2.0',
    'status': 'pending',
})

print(f'Step 1: Created event {event_id}')
print(f'Step 2: Would launch: python scripts/train.py --event-id {event_id}')

Step 1: Created event train_20260318_091620
Step 2: Would launch: python scripts/train.py --event-id train_20260318_091620


### Step 2: scripts/train.py reuses event_id

In [20]:
# Simulating scripts/train.py behavior

db.mark_training_event_running(event_id)
print(f'Step 3: Marked event as running')

publisher = get_pipeline_event_publisher({'events': {'backend': 'noop'}})
publisher.publish('training.started', {'event_id': event_id})
print(f'Step 4: Published training.started event')

Step 3: Marked event as running
Step 4: Published training.started event


### Step 3: Training completes

In [21]:
# Simulating successful training completion

db.mark_training_event_completed(event_id)
print(f'Step 5: Marked event as completed')

publisher.publish('training.completed', {'event_id': event_id})
print(f'Step 6: Published training.completed event')

events = db.get_training_events(limit=1)
print(f'\nFinal status: {events[0]["status"]}')
print(f'Event ID: {events[0]["event_id"]}')

Step 5: Marked event as completed
Step 6: Published training.completed event

Final status: completed
Event ID: train_20260318_091620


---

## Part 8: Testing & Verification

### Unit Tests for Event Lifecycle

In [22]:
# Example test from tests/test_training_events.py

import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    db_path = f'{tmpdir}/test.db'
    db = AnalyticsDB(db_path)
    
    event_id = db.save_training_event({
        'trigger_type': 'manual',
        'dataset_size': 10,
        'drift_score': 0.0,
        'model_version': 'v1',
        'status': 'pending',
    })
    
    assert db.mark_training_event_running(event_id, dataset_size=25)
    assert db.mark_training_event_completed(event_id)
    
    events = db.get_training_events(limit=1)
    assert events[0]['status'] == 'completed'
    assert events[0]['dataset_size'] == 25
    
    print('✓ All assertions passed')

✓ All assertions passed


PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\thoma\\AppData\\Local\\Temp\\tmpbgb7p1qa\\test.db'

---

## Summary

### Key Takeaways

1. **Event Deduplication**: Pass event IDs via CLI to prevent duplicate logging
2. **Lifecycle Tracking**: Use state machine (pending → running → completed/failed)
3. **Robust Ingestion**: Dedupe, validate, and merge multiple sources
4. **Abstraction**: Use interfaces for extensibility (Kafka-ready)
5. **SOLID Principles**: Apply SRP, OCP, LSP, ISP, DIP throughout

### Files Modified

- `src/vision_ml/analytics/analytics_db.py`: Event lifecycle methods
- `src/vision_ml/events/publishers.py`: Event publisher abstraction
- `scripts/prepare_data.py`: Robust ingestion with dedup/validation
- `scripts/train.py`: Event ID reuse, lifecycle updates, event publishing
- `pages/4_training.py`, `pages/7_training_pipeline.py`: Pass event IDs to scripts
- `tests/test_training_events.py`, `tests/test_prepare_data_ingestion.py`: Unit tests

### Next Steps

1. Run full test suite: `pytest tests/`
2. Test ingestion: `python scripts/prepare_data.py --source local`
3. Test training: `python scripts/train.py --epochs 1 --device cpu`
4. Monitor events in Streamlit dashboard